# PARTIE 2 — Génération de données synthétiques & Confidentialité
## Version AVEC packages (ctgan, sdv, pgmpy, diffprivlib)

Ce notebook fait suite à l'EDA et au preprocessing du notebook principal.
Il couvre :
1. Préparation des données pour la synthèse
2. CTGAN (GAN tabulaire)
3. TVAE (Variational Autoencoder tabulaire)
4. Modèle Bayésien (BayesianNetwork via pgmpy)
5. Confidentialité : k-anonymat, l-diversity, Differential Privacy
6. Évaluation comparative (TSTR / TRTS)

## 0. Installation des packages

In [ ]:
# ⚠️ À exécuter une seule fois dans Colab
# Temps estimé : 3-5 minutes
!pip install ctgan sdv pgmpy diffprivlib -q
!pip install scikit-learn xgboost -q

## 1. Imports & Chargement des données

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

print('✅ Imports OK')

In [ ]:
# Chargement de la base originale
app_train = pd.read_csv('./sample_data/application_train.csv')
print(f'Shape original : {app_train.shape}')
print(f'Taux de défaut : {app_train["TARGET"].mean():.2%}')

## 2. Préparation des données pour la synthèse

In [ ]:
# -------------------------------------------------------
# SÉLECTION DES VARIABLES
# On garde un sous-ensemble représentatif pour la synthèse
# (les 122 colonnes sont trop lourdes pour les GAN en dev)
# -------------------------------------------------------

COLS_NUMERIQUES = [
    'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE',
    'DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_REGISTRATION', 'DAYS_ID_PUBLISH',
    'CNT_CHILDREN', 'CNT_FAM_MEMBERS',
    'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3'
]

COLS_CATEGORIELLES = [
    'NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY',
    'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS',
    'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE', 'WEEKDAY_APPR_PROCESS_START'
]

CIBLE = 'TARGET'

TOUTES_COLS = COLS_NUMERIQUES + COLS_CATEGORIELLES + [CIBLE]

# Sous-ensemble propre
df = app_train[TOUTES_COLS].copy()
print(f'Shape après sélection : {df.shape}')

In [ ]:
# -------------------------------------------------------
# NETTOYAGE : valeurs aberrantes connues sur DAYS_EMPLOYED
# La valeur 365243 est un code pour "non applicable"
# -------------------------------------------------------
df['DAYS_EMPLOYED'].replace(365243, np.nan, inplace=True)

# DAYS_BIRTH est négatif (jours avant aujourd'hui) → on le convertit en âge positif
df['AGE_YEARS'] = -df['DAYS_BIRTH'] / 365
df['YEARS_EMPLOYED'] = -df['DAYS_EMPLOYED'] / 365
df.drop(columns=['DAYS_BIRTH', 'DAYS_EMPLOYED'], inplace=True)

# Mise à jour des listes
COLS_NUMERIQUES = [c for c in COLS_NUMERIQUES if c not in ['DAYS_BIRTH','DAYS_EMPLOYED']]
COLS_NUMERIQUES += ['AGE_YEARS', 'YEARS_EMPLOYED']

print(f'Shape après nettoyage : {df.shape}')

In [ ]:
# -------------------------------------------------------
# IMPUTATION des valeurs manquantes
# -------------------------------------------------------

# Numériques → médiane
imputer_num = SimpleImputer(strategy='median')
df[COLS_NUMERIQUES] = imputer_num.fit_transform(df[COLS_NUMERIQUES])

# Catégorielles → mode (most_frequent)
imputer_cat = SimpleImputer(strategy='most_frequent')
df[COLS_CATEGORIELLES] = imputer_cat.fit_transform(df[COLS_CATEGORIELLES])

print(f'Valeurs manquantes restantes : {df.isnull().sum().sum()}')

In [ ]:
# -------------------------------------------------------
# ÉCHANTILLON pour le développement
# Pour la version finale : supprimer cette cellule
# et utiliser df complet
# -------------------------------------------------------

# ⚡ RECOMMANDATION : 50k lignes pour dev rapide
# Commenter la ligne ci-dessous pour utiliser tout le dataset
df_sample = df.sample(n=50_000, random_state=42).reset_index(drop=True)

# Pour la version COMPLÈTE décommenter :
# df_sample = df.copy()

print(f'Taille du jeu utilisé : {df_sample.shape}')
print(f'Taux de défaut : {df_sample[CIBLE].mean():.2%}')

## 3. CTGAN — Conditional Tabular GAN

**Principe** : CTGAN est un GAN (Generative Adversarial Network) adapté aux données tabulaires.
Il utilise un *conditional vector* pour gérer le déséquilibre des classes et les variables catégorielles.
Le générateur produit des données synthétiques que le discriminateur tente de distinguer des vraies données.

**Temps estimé (50k lignes, 100 epochs)** : ~15–30 min sur CPU Colab

In [ ]:
from ctgan import CTGAN

# -------------------------------------------------------
# CONFIGURATION CTGAN
# -------------------------------------------------------
ctgan_model = CTGAN(
    epochs=100,              # Augmenter à 300 pour version finale
    batch_size=500,
    generator_dim=(256, 256),
    discriminator_dim=(256, 256),
    verbose=True             # Affiche la loss à chaque epoch
)

# Spécifier les colonnes catégorielles (obligatoire pour CTGAN)
ctgan_model.fit(
    df_sample,
    discrete_columns=COLS_CATEGORIELLES + [CIBLE]
)

print('✅ CTGAN entraîné')

In [ ]:
# Génération des données synthétiques CTGAN
# On génère le même nombre de lignes que l'échantillon d'origine
df_ctgan = ctgan_model.sample(len(df_sample))

print(f'Shape CTGAN synthétique : {df_ctgan.shape}')
print(f'Taux de défaut CTGAN    : {df_ctgan[CIBLE].mean():.2%}')
print(f'Taux de défaut réel     : {df_sample[CIBLE].mean():.2%}')

In [ ]:
# -------------------------------------------------------
# VISUALISATION : comparaison distributions
# -------------------------------------------------------
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
cols_viz = ['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AGE_YEARS',
            'EXT_SOURCE_2', 'YEARS_EMPLOYED', 'CNT_CHILDREN']

for ax, col in zip(axes.flatten(), cols_viz):
    sns.kdeplot(df_sample[col], ax=ax, label='Réel', color='steelblue', fill=True, alpha=0.4)
    sns.kdeplot(df_ctgan[col], ax=ax, label='CTGAN', color='tomato', fill=True, alpha=0.4)
    ax.set_title(col)
    ax.legend()

plt.suptitle('CTGAN — Comparaison des distributions', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 4. TVAE — Tabular Variational Autoencoder

**Principe** : TVAE encode les données dans un espace latent continu (distribution gaussienne),
puis les décode pour générer de nouvelles observations.
Plus stable que CTGAN à l'entraînement, mais parfois moins fidèle sur les distributions extrêmes.

**Temps estimé (50k lignes)** : ~10–20 min sur CPU Colab

In [ ]:
from sdv.single_table import TVAESynthesizer
from sdv.metadata import SingleTableMetadata

# -------------------------------------------------------
# MÉTADONNÉES : SDV a besoin de connaître les types
# -------------------------------------------------------
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(df_sample)

# Correction manuelle des types si nécessaire
for col in COLS_CATEGORIELLES + [CIBLE]:
    metadata.update_column(col, sdtype='categorical')

print('Metadata détectées :')
metadata.visualize()  # Affiche un résumé visuel dans Colab

In [ ]:
# -------------------------------------------------------
# CONFIGURATION TVAE
# -------------------------------------------------------
tvae_model = TVAESynthesizer(
    metadata,
    epochs=100,              # Augmenter à 300 pour version finale
    batch_size=500,
    compress_dims=(128, 128),
    decompress_dims=(128, 128),
    embedding_dim=128,
    verbose=True
)

tvae_model.fit(df_sample)
print('✅ TVAE entraîné')

In [ ]:
# Génération des données synthétiques TVAE
df_tvae = tvae_model.sample(num_rows=len(df_sample))

print(f'Shape TVAE synthétique : {df_tvae.shape}')
print(f'Taux de défaut TVAE    : {df_tvae[CIBLE].mean():.2%}')
print(f'Taux de défaut réel    : {df_sample[CIBLE].mean():.2%}')

In [ ]:
# Visualisation TVAE vs Réel
fig, axes = plt.subplots(2, 3, figsize=(16, 8))

for ax, col in zip(axes.flatten(), cols_viz):
    sns.kdeplot(df_sample[col], ax=ax, label='Réel', color='steelblue', fill=True, alpha=0.4)
    sns.kdeplot(df_tvae[col], ax=ax, label='TVAE', color='green', fill=True, alpha=0.4)
    ax.set_title(col)
    ax.legend()

plt.suptitle('TVAE — Comparaison des distributions', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 5. Modèle Bayésien — Bayesian Network

**Principe** : Un réseau bayésien apprend la structure de dépendances conditionnelles
entre variables sous forme de DAG (graphe acyclique dirigé).
Il modélise P(X1, X2, ..., Xn) en décomposant selon les dépendances.
C'est un bon benchmark car il est explicable et repose sur des hypothèses probabilistes claires.

**Temps estimé (50k lignes)** : ~5–15 min sur CPU Colab

On utilise ici **pgmpy** qui offre l'algorithme HillClimbing pour apprendre la structure.

In [ ]:
from pgmpy.models import BayesianNetwork
from pgmpy.estimators import HillClimbSearch, BicScore, BayesianEstimator
from pgmpy.sampling import BayesianModelSampling

# -------------------------------------------------------
# PRÉPARATION : le réseau bayésien nécessite des variables
# discrètes — on discrétise les numériques en quantiles
# -------------------------------------------------------

# On travaille sur un sous-échantillon encore plus réduit
# car la structure learning est O(n²) en nombre de variables
N_BN = 10_000
df_bn = df_sample.sample(n=N_BN, random_state=42).copy()

# Variables sélectionnées pour le réseau bayésien
# (moins de variables pour que la structure learning soit faisable)
COLS_BN_NUM = ['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AGE_YEARS',
               'EXT_SOURCE_2', 'YEARS_EMPLOYED']
COLS_BN_CAT = ['CODE_GENDER', 'NAME_EDUCATION_TYPE', 'NAME_INCOME_TYPE']

df_bn_disc = df_bn[COLS_BN_NUM + COLS_BN_CAT + [CIBLE]].copy()

# Discrétisation des variables numériques en quartiles
for col in COLS_BN_NUM:
    df_bn_disc[col] = pd.qcut(df_bn_disc[col], q=4, labels=['Q1','Q2','Q3','Q4'], duplicates='drop')
    df_bn_disc[col] = df_bn_disc[col].astype(str)

# Conversion des catégorielles en str
for col in COLS_BN_CAT + [CIBLE]:
    df_bn_disc[col] = df_bn_disc[col].astype(str)

print(f'Shape pour Bayesian Network : {df_bn_disc.shape}')
df_bn_disc.head(3)

In [ ]:
# -------------------------------------------------------
# APPRENTISSAGE DE LA STRUCTURE (Hill Climbing + BIC)
# -------------------------------------------------------
print('Apprentissage de la structure... (peut prendre 5-10 min)')

hc = HillClimbSearch(df_bn_disc)
best_structure = hc.estimate(scoring_method=BicScore(df_bn_disc),
                              max_iter=200)

print(f'Structure apprise : {best_structure.edges()}')

In [ ]:
# -------------------------------------------------------
# APPRENTISSAGE DES PARAMÈTRES (distributions conditionnelles)
# -------------------------------------------------------
bn_model = BayesianNetwork(best_structure.edges())
bn_model.fit(df_bn_disc, estimator=BayesianEstimator, prior_type='BDeu', equivalent_sample_size=10)

print('✅ Réseau bayésien entraîné')
print(f'Nombre de noeuds : {len(bn_model.nodes())}')
print(f'Nombre d\'arêtes : {len(bn_model.edges())}')

In [ ]:
# -------------------------------------------------------
# GÉNÉRATION des données synthétiques Bayésiennes
# -------------------------------------------------------
inference = BayesianModelSampling(bn_model)
df_bayes = inference.forward_sample(size=N_BN)

print(f'Shape Bayésien synthétique : {df_bayes.shape}')
print(f'Taux de défaut Bayésien    : {(df_bayes[CIBLE] == "1").mean():.2%}')

In [ ]:
# Visualisation de la structure du réseau bayésien
try:
    import networkx as nx
    G = nx.DiGraph(bn_model.edges())
    plt.figure(figsize=(10, 7))
    pos = nx.spring_layout(G, seed=42)
    nx.draw(G, pos, with_labels=True, node_size=2000,
            node_color='lightblue', font_size=8, arrows=True,
            arrowsize=20, edge_color='gray')
    plt.title('Structure du Réseau Bayésien apprise sur Home Credit')
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f'Visualisation réseau : {e}')

## 6. Confidentialité

### 6a. k-anonymat
**Définition** : Un dataset est k-anonyme si chaque individu est indiscernable
d'au moins k-1 autres individus sur les quasi-identifiants.

**Quasi-identifiants** (QI) dans notre contexte crédit : âge, genre, type de revenu, éducation

In [ ]:
# -------------------------------------------------------
# k-ANONYMAT
# -------------------------------------------------------

def compute_k_anonymity(df, quasi_identifiers):
    """
    Calcule le k-anonymat d'un DataFrame.
    Retourne le k minimal (pire cas) et la distribution des groupes.
    """
    groups = df.groupby(quasi_identifiers).size()
    k_min = groups.min()
    return k_min, groups


# Quasi-identifiants
QI = ['CODE_GENDER', 'NAME_EDUCATION_TYPE', 'NAME_INCOME_TYPE']

# Pour cette analyse, on travaille avec les colonnes catégorielles
# On crée une version discrétisée de AGE pour l'inclure
df_privacy = df_sample[QI + [CIBLE]].copy()
df_ctgan_privacy = df_ctgan[QI + [CIBLE]].copy()

# k-anonymat sur données réelles
k_reel, groups_reel = compute_k_anonymity(df_privacy, QI)

# k-anonymat sur données CTGAN
k_ctgan, groups_ctgan = compute_k_anonymity(df_ctgan_privacy, QI)

print('=== k-ANONYMAT ===')
print(f'Données réelles  : k_min = {k_reel}')
print(f'Données CTGAN    : k_min = {k_ctgan}')
print()
print('Distribution des tailles de groupes (données réelles) :')
print(groups_reel.describe())

In [ ]:
# Visualisation k-anonymat
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

groups_reel.hist(bins=30, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title(f'Réel — Distribution des groupes (k_min={k_reel})')
axes[0].set_xlabel('Taille du groupe')
axes[0].set_ylabel('Fréquence')

groups_ctgan.hist(bins=30, ax=axes[1], color='tomato', edgecolor='white')
axes[1].set_title(f'CTGAN — Distribution des groupes (k_min={k_ctgan})')
axes[1].set_xlabel('Taille du groupe')

plt.tight_layout()
plt.show()

### 6b. l-diversity

**Définition** : Extension du k-anonymat. Un groupe est l-diverse si l'attribut sensible
(ici TARGET = défaut) prend au moins l valeurs distinctes dans ce groupe.
Cela protège contre les attaques d'homogénéité.

In [ ]:
# -------------------------------------------------------
# l-DIVERSITY
# -------------------------------------------------------

def compute_l_diversity(df, quasi_identifiers, sensitive_col):
    """
    Calcule la l-diversity d'un DataFrame.
    Retourne le l minimal (pire cas) et les statistiques.
    """
    # Pour chaque groupe QI, compter le nombre de valeurs distinctes de l'attribut sensible
    l_values = df.groupby(quasi_identifiers)[sensitive_col].nunique()
    l_min = l_values.min()
    return l_min, l_values


# l-diversity sur données réelles
l_reel, l_dist_reel = compute_l_diversity(df_privacy, QI, CIBLE)

# l-diversity sur données CTGAN
l_ctgan, l_dist_ctgan = compute_l_diversity(df_ctgan_privacy, QI, CIBLE)

print('=== l-DIVERSITY ===')
print(f'Données réelles  : l_min = {l_reel}')
print(f'Données CTGAN    : l_min = {l_ctgan}')
print()
print('Groupes avec l = 1 dans les données réelles (risque !) :')
groupes_risques = l_dist_reel[l_dist_reel == 1]
print(f'  Nombre : {len(groupes_risques)} sur {len(l_dist_reel)} groupes')
print(f'  Soit {len(groupes_risques)/len(l_dist_reel):.1%} des groupes')

In [ ]:
# Résumé visuel k-anonymat vs l-diversity
print('\n=== RÉSUMÉ CONFIDENTIALITÉ ===')
print(f"{'Métrique':<25} {'Données Réelles':>18} {'Données CTGAN':>15}")
print('-' * 60)
print(f"{'k-anonymat (k_min)':<25} {k_reel:>18} {k_ctgan:>15}")
print(f"{'l-diversity (l_min)':<25} {l_reel:>18} {l_ctgan:>15}")
print(f"{'Nbre de groupes QI':<25} {len(groups_reel):>18} {len(groups_ctgan):>15}")

### 6c. Differential Privacy (ε-DP)

**Principe** : La confidentialité différentielle garantit mathématiquement qu'aucun individu
ne peut être identifié dans le dataset. On ajoute du bruit calibré (mécanisme de Laplace ou Gaussien)
aux statistiques ou aux données.

**ε (epsilon)** : Plus ε est petit, plus la protection est forte mais les données moins précises.
- ε = 0.1 → très protégé, peu utile
- ε = 1.0 → bon compromis
- ε = 10.0 → peu protégé, très utile

In [ ]:
from diffprivlib import tools as dp
import diffprivlib as dpl

# -------------------------------------------------------
# STATISTIQUES AVEC DIFFERENTIAL PRIVACY
# -------------------------------------------------------

EPSILON_VALUES = [0.1, 0.5, 1.0, 5.0, 10.0]

col_dp = 'AMT_INCOME_TOTAL'
true_mean = df_sample[col_dp].mean()
true_std = df_sample[col_dp].std()

print(f'Statistiques réelles pour {col_dp}:')
print(f'  Moyenne vraie : {true_mean:,.0f}')
print(f'  Écart-type vrai : {true_std:,.0f}')
print()
print(f'=== IMPACT DE ε SUR LES STATISTIQUES DP ===')
print(f"{'ε':<8} {'Moyenne DP':>15} {'Erreur relative':>18}")
print('-' * 45)

bounds = (df_sample[col_dp].min(), df_sample[col_dp].max())

for eps in EPSILON_VALUES:
    mean_dp = dp.mean(df_sample[col_dp].values, epsilon=eps, bounds=bounds)
    erreur = abs(mean_dp - true_mean) / true_mean
    print(f"ε={eps:<5} {mean_dp:>15,.0f} {erreur:>17.2%}")

In [ ]:
# -------------------------------------------------------
# DONNÉES SYNTHÉTIQUES AVEC DP : ajout de bruit gaussien
# sur les colonnes numériques des données CTGAN
# (approche DP post-processing)
# -------------------------------------------------------

def apply_dp_noise(df_synthetic, numeric_cols, epsilon=1.0):
    """
    Applique un bruit Laplacien sur les colonnes numériques
    des données synthétiques pour renforcer la DP.
    Sensibilité = range / n
    """
    df_dp = df_synthetic.copy()
    for col in numeric_cols:
        col_range = df_dp[col].max() - df_dp[col].min()
        sensitivity = col_range / len(df_dp)
        scale = sensitivity / epsilon
        noise = np.random.laplace(loc=0, scale=scale, size=len(df_dp))
        df_dp[col] = df_dp[col] + noise
    return df_dp


# Application avec ε = 1.0 (bon compromis utilité/confidentialité)
COLS_NUM_CTGAN = [c for c in COLS_NUMERIQUES if c in df_ctgan.columns]
df_ctgan_dp = apply_dp_noise(df_ctgan, COLS_NUM_CTGAN, epsilon=1.0)

print(f'✅ Données CTGAN + DP générées (ε=1.0)')
print(f'Moyenne AMT_INCOME_TOTAL réelle : {df_sample["AMT_INCOME_TOTAL"].mean():,.0f}')
print(f'Moyenne CTGAN                   : {df_ctgan["AMT_INCOME_TOTAL"].mean():,.0f}')
print(f'Moyenne CTGAN + DP (ε=1.0)      : {df_ctgan_dp["AMT_INCOME_TOTAL"].mean():,.0f}')

## 7. Évaluation comparative — TSTR / TRTS

**TSTR (Train on Synthetic, Test on Real)** : entraîner un modèle sur synthétique, tester sur réel.
Si AUC ≈ TRTR (baseline), les données synthétiques preservent le signal prédictif.

**TRTS (Train on Real, Test on Synthetic)** : inverse — vérifie si le synthétique
est statistiquement cohérent avec le réel.

In [ ]:
from sklearn.preprocessing import LabelEncoder

def prepare_for_model(df, cat_cols, num_cols, target):
    """Encode les catégorielles et retourne X, y."""
    df_enc = df.copy()
    le = LabelEncoder()
    for col in cat_cols:
        df_enc[col] = le.fit_transform(df_enc[col].astype(str))
    X = df_enc[num_cols + cat_cols].values
    y = df_enc[target].values.astype(int)
    return X, y


def evaluate_tstr(df_train_synth, df_test_real, cat_cols, num_cols, target):
    """TSTR : Train Synthetic → Test Real"""
    X_train, y_train = prepare_for_model(df_train_synth, cat_cols, num_cols, target)
    X_test, y_test = prepare_for_model(df_test_real, cat_cols, num_cols, target)

    model = LogisticRegression(max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    y_proba = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_proba)
    return auc


# Sélection des colonnes communes
COLS_COMMUNES_NUM = [c for c in COLS_NUMERIQUES if c in df_ctgan.columns and c in df_tvae.columns]
COLS_COMMUNES_CAT = [c for c in COLS_CATEGORIELLES if c in df_ctgan.columns and c in df_tvae.columns]

# Split du réel pour test
df_reel_train, df_reel_test = train_test_split(df_sample, test_size=0.2, random_state=42)

print('Calcul des AUC TSTR... (peut prendre 1-2 min)')

# TRTR : baseline — entraîner ET tester sur réel
auc_trtr = evaluate_tstr(df_reel_train, df_reel_test, COLS_COMMUNES_CAT, COLS_COMMUNES_NUM, CIBLE)

# TSTR CTGAN
auc_ctgan = evaluate_tstr(df_ctgan, df_reel_test, COLS_COMMUNES_CAT, COLS_COMMUNES_NUM, CIBLE)

# TSTR TVAE
auc_tvae = evaluate_tstr(df_tvae, df_reel_test, COLS_COMMUNES_CAT, COLS_COMMUNES_NUM, CIBLE)

print('\n=== RÉSULTATS TSTR (AUC Logistic Regression) ===')
print(f"{'Modèle':<20} {'AUC':>10} {'Delta vs baseline':>20}")
print('-' * 52)
print(f"{'TRTR (baseline)':<20} {auc_trtr:>10.4f} {'—':>20}")
print(f"{'TSTR CTGAN':<20} {auc_ctgan:>10.4f} {auc_ctgan - auc_trtr:>+20.4f}")
print(f"{'TSTR TVAE':<20} {auc_tvae:>10.4f} {auc_tvae - auc_trtr:>+20.4f}")

In [ ]:
# Visualisation finale : tableau de bord comparatif
results = {
    'Modèle': ['TRTR (Réel)', 'CTGAN', 'TVAE'],
    'AUC TSTR': [auc_trtr, auc_ctgan, auc_tvae],
    'k_min': [k_reel, k_ctgan, None],
    'l_min': [l_reel, l_ctgan, None]
}

df_results = pd.DataFrame(results)
print('\n=== TABLEAU COMPARATIF FINAL ===')
print(df_results.to_string(index=False))

# Bar chart AUC
plt.figure(figsize=(8, 5))
colors = ['steelblue', 'tomato', 'green']
bars = plt.bar(df_results['Modèle'], df_results['AUC TSTR'], color=colors)
plt.axhline(y=auc_trtr, color='navy', linestyle='--', label='Baseline réel')
plt.ylim(0.5, 1.0)
plt.ylabel('AUC')
plt.title('Comparaison AUC TSTR — Modèles génératifs')
plt.legend()
for bar, val in zip(bars, df_results['AUC TSTR']):
    plt.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
             f'{val:.4f}', ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Sauvegarde des données synthétiques

In [ ]:
# Sauvegarde des jeux de données synthétiques
df_ctgan.to_csv('./sample_data/synthetic_ctgan.csv', index=False)
df_tvae.to_csv('./sample_data/synthetic_tvae.csv', index=False)
df_ctgan_dp.to_csv('./sample_data/synthetic_ctgan_dp.csv', index=False)

print('✅ Données synthétiques sauvegardées :')
print('  - synthetic_ctgan.csv')
print('  - synthetic_tvae.csv')
print('  - synthetic_ctgan_dp.csv (avec Differential Privacy ε=1.0)')